# RealSaS — Stage-B7 TRAIN512 Atomic Promotion V2

**Run All. CPU is sufficient. Zero DINO optimizer steps.**

This notebook promotes only the already-qualified nine TRAIN512 repaired payloads from the sealed Stage-B7 staging tree into their existing canonical asset/variant/consumer paths.

Safety contract:
- requires exact Stage-B7 staging PASS content SHA;
- reconstructs the exact historical TRAIN512 set before mutation;
- takes a full backup of all 9 affected canonical asset/variant/export subtrees before the first write;
- global selection / MASTER_VARIANTS / PROCESSED_VARIANTS / consumer records are never written and must remain byte-exact;
- any error restores all 9 affected canonical paths;
- post-promotion TRAIN512 availability must be 512/512;
- no DINO feature extraction or training occurs.


In [ ]:
# 0) Drive + fail-closed authority checks
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import hashlib, json, sys

MASTER = Path("/content/drive/MyDrive/RealSaS_MASTER_CORPUS_1024_V3")
POST = MASTER / "reports" / "post_corpus_audit"
RUNNER = POST / "stage_b7_train512_atomic_promotion_v2.py"
STAGE_RESULT = POST / "B7_TRAIN512_SELECTIVE_REPAIR_STAGING_V2" / "STAGE_B7_RESULT_V2.json"
EXPECTED_RUNNER_SHA = "f9e49dd8b35f22ff5b98b6657b9e01694dc556c148e1affb99808ecdfe1915ac"
EXPECTED_STAGE_CONTENT_SHA = "13bee59ea268e232569e041710b52b28ba3900f98e676fc55c36b48ecba85447"

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(8<<20),b""): h.update(b)
    return h.hexdigest()

print("="*88, flush=True)
print("[PROMOTE] PREFLIGHT", flush=True)
print("[runtime] python:", sys.executable, sys.version.replace("\n"," "), flush=True)
print("[path] master:", MASTER, MASTER.is_dir(), flush=True)
print("[path] runner:", RUNNER, RUNNER.is_file(), flush=True)
print("[path] stage result:", STAGE_RESULT, STAGE_RESULT.is_file(), flush=True)
if not MASTER.is_dir() or not RUNNER.is_file() or not STAGE_RESULT.is_file():
    raise RuntimeError("PROMOTION_REQUIRED_PATH_MISSING")

got=sha256_file(RUNNER)
print("[runner] sha256:", got, flush=True)
if got != EXPECTED_RUNNER_SHA:
    raise RuntimeError(f"RUNNER_SHA_DRIFT expected={EXPECTED_RUNNER_SHA} actual={got}")

sr=json.loads(STAGE_RESULT.read_text())
print("[staging] status:", sr.get("status"), flush=True)
print("[staging] content_sha256:", sr.get("content_sha256"), flush=True)
print("[staging] sentinel_pass:", (sr.get("sentinel") or {}).get("pass"), flush=True)
print("[staging] repair pass:", sr.get("pass_asset_count"), "/", sr.get("staged_asset_count"), flush=True)
if sr.get("status") != "PASS_STAGING_ONLY__MASTER_UNCHANGED": raise RuntimeError("STAGING_NOT_PASS")
if sr.get("content_sha256") != EXPECTED_STAGE_CONTENT_SHA: raise RuntimeError("STAGING_CONTENT_SHA_DRIFT")
if not (sr.get("sentinel") or {}).get("pass") or sr.get("pass_asset_count") != 9: raise RuntimeError("STAGING_GATE_DRIFT")
print("[PROMOTE] PREFLIGHT: PASS", flush=True)


In [ ]:
# 1) Atomic canonical promotion + post-promotion TRAIN512 512/512 audit
import subprocess, sys

print("="*88, flush=True)
print("[PROMOTE] EXECUTING SEALED ATOMIC PROMOTION", flush=True)
print("[PROMOTE] full backup occurs before first canonical write", flush=True)
print("[PROMOTE] global ledgers/selection/records must remain byte-exact", flush=True)
print("="*88, flush=True)

cmd=[sys.executable, str(RUNNER), "--root", str(MASTER)]
p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
rc=p.wait()
print("[PROMOTE] runner_exit_code:", rc, flush=True)

RESULT=POST/"B7_TRAIN512_SELECTIVE_PROMOTION_RESULT_V2.json"
AUTH=POST/"DINO_TRAIN512_NATIVE1024_AUTHORITY_V2.json"
if RESULT.is_file():
    r=json.loads(RESULT.read_text())
    print("="*88, flush=True)
    print("[PROMOTE] FINAL RESULT", flush=True)
    print(json.dumps({
        "status":r.get("status"),
        "promoted_asset_count":r.get("promoted_asset_count"),
        "train512_ready_count":r.get("train512_ready_count"),
        "immutable_globals_unchanged":r.get("immutable_globals_unchanged"),
        "master_ledger_sha256_before":r.get("master_ledger_sha256_before"),
        "master_ledger_sha256_after":r.get("master_ledger_sha256_after"),
        "scientific_optimizer_steps":r.get("scientific_optimizer_steps"),
        "content_sha256":r.get("content_sha256"),
    },indent=2,sort_keys=True),flush=True)
    print("[PROMOTE] RESULT PATH:",RESULT,flush=True)
    print("[PROMOTE] TRAIN512 AUTHORITY PATH:",AUTH,flush=True)
    print("="*88,flush=True)
if rc != 0:
    raise RuntimeError("ATOMIC_PROMOTION_FAILED__SEE_STDOUT_AND_DRIVE_RESULT")
if not RESULT.is_file(): raise RuntimeError("PROMOTION_RESULT_MISSING")
if r.get("status") != "PASS_PROMOTED_9__TRAIN512_NATIVE1024_512_OF_512": raise RuntimeError("PROMOTION_NOT_PASS")
if r.get("train512_ready_count") != 512 or r.get("immutable_globals_unchanged") is not True: raise RuntimeError("PROMOTION_FINAL_GATE_DRIFT")
print("[PROMOTE] COMPLETE: PASS_PROMOTED_9__TRAIN512_NATIVE1024_512_OF_512",flush=True)
print("[PROMOTE] DINO training still NOT started; next zero-step preflight gates remain.",flush=True)
